# Logistic regression

We compare L2-regularized logistic regression with the majority baseline from `02_baseline.ipynb`. This first experiment uses the existing random split and excludes call duration. The positive class is a subscription (`y = 1`).

Run from the repository root or `src/`. The experiment function below takes prepared train, validation, and test arrays, so it can also be used with a different split or feature set. For every new split, fit preprocessing on that split's training rows only.

In [14]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

cwd = Path.cwd()
root = cwd.parent if cwd.name == "src" else cwd
sys.path.insert(0, str(root / "src"))
from model_utils import evaluate, load_random_split, validate_splits

RANDOM_STATE = 42
THRESHOLD = 0.5
C_VALUES = (0.01, 0.1, 1.0, 10.0)

## Training and evaluation

Each candidate is trained on the training set. We choose C using validation average precision (PR-AUC); ties go to the smaller C. The selected model is not refitted on validation, and test scores do not affect selection. Classification metrics use a fixed 0.5 threshold.

The function also accepts `selection_metric="roc_auc"` for a future experiment that calls for it. ROC-AUC requires both classes in each evaluated split; a split without both classes raises an error rather than reporting a misleading value.

In [15]:
def run_experiment(splits, split_scheme, feature_set, selection_metric="average_precision"):
    """Fit and compare models using prepared (X, y) pairs for train, val, and test."""
    if selection_metric not in ("average_precision", "roc_auc"):
        raise ValueError("selection_metric must be average_precision or roc_auc")
    validate_splits(splits)
    X_train, y_train = splits["train"]
    X_val, y_val = splits["val"]

    candidates = {}
    validation_rows = []
    for c in C_VALUES:
        model = LogisticRegression(C=c, solver="lbfgs", max_iter=3000, random_state=RANDOM_STATE)
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            model.fit(X_train, y_train)
        candidates[c] = model
        validation_rows.append({"C": c, **evaluate(model, X_val, y_val, THRESHOLD)})

    validation = pd.DataFrame(validation_rows)
    best_c = float(validation.sort_values([selection_metric, "C"], ascending=[False, True]).iloc[0]["C"])
    selected = candidates[best_c]
    baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

    rows = []
    for name in ("train", "val", "test"):
        X, y = splits[name]
        rows.append({"model": "Majority", "role": "baseline", "C": np.nan, "split": name,
                     **evaluate(baseline, X, y, THRESHOLD)})
        rows.append({"model": "Logistic regression", "role": "selected", "C": best_c, "split": name,
                     **evaluate(selected, X, y, THRESHOLD)})
    for row in validation_rows:
        if row["C"] != best_c:
            rows.append({"model": "Logistic regression", "role": "candidate", "split": "val", **row})

    results = pd.DataFrame(rows)
    results.insert(0, "threshold", THRESHOLD)
    results.insert(0, "selection_metric", selection_metric)
    results.insert(0, "features", feature_set)
    results.insert(0, "split_scheme", split_scheme)
    return selected, best_c, results

## Random split, without duration

The table includes the majority baseline and selected model on all three sets, plus the other candidates on validation. `candidate` rows were used for selection; `selected` rows show the same fitted model across splits. Average precision and ROC-AUC use probabilities, while accuracy, precision, recall, and F1 use the fixed threshold.

In [16]:
splits = load_random_split(root / "data" / "processed" / "arrays.npz")
selected_model, best_c, results = run_experiment(
    splits, split_scheme="random", feature_set="without duration"
)
print(f"Selected C: {best_c:g} (highest validation average precision)")
if best_c in (min(C_VALUES), max(C_VALUES)):
    print("The selected C is at the edge of the tested grid; the search is limited.")
display(results.round(4))

Selected C: 10 (highest validation average precision)
The selected C is at the edge of the tested grid; the search is limited.


,split_scheme,features,selection_metric,threshold,model,role,C,split,n,positive_rate,average_precision,roc_auc,accuracy,precision,recall,f1
0,random,without duration,average_precision,0.5,Majority,baseline,NaN,train,28822,0.1127,0.1127,0.5000,0.8873,0.0000,0.0000,0.0000
1,random,without duration,average_precision,0.5,Logistic regression,selected,10.00,train,28822,0.1127,0.4589,0.7947,0.9005,0.6673,0.2322,0.3445
2,random,without duration,average_precision,0.5,Majority,baseline,NaN,val,6177,0.1127,0.1127,0.5000,0.8873,0.0000,0.0000,0.0000
3,random,without duration,average_precision,0.5,Logistic regression,selected,10.00,val,6177,0.1127,0.4447,0.7871,0.9004,0.6709,0.2284,0.3408
4,random,without duration,average_precision,0.5,Majority,baseline,NaN,test,6177,0.1127,0.1127,0.5000,0.8873,0.0000,0.0000,0.0000
5,random,without duration,average_precision,0.5,Logistic regression,selected,10.00,test,6177,0.1127,0.4577,0.8017,0.9004,0.6800,0.2198,0.3322
6,random,without duration,average_precision,0.5,Logistic regression,candidate,0.01,val,6177,0.1127,0.4409,0.7849,0.9009,0.7039,0.2083,0.3215
7,random,without duration,average_precision,0.5,Logistic regression,candidate,0.10,val,6177,0.1127,0.4428,0.7868,0.9001,0.6653,0.2284,0.3401
8,random,without duration,average_precision,0.5,Logistic regression,candidate,1.00,val,6177,0.1127,0.4438,0.7871,0.9004,0.6723,0.2270,0.3394


## Validation errors

Rows are actual classes and columns are predicted classes. A false negative is a subscriber we missed; a false positive is a non-subscriber predicted to subscribe. These counts describe the 0.5 threshold, not the ranking used to select C.

In [17]:
X_val, y_val = splits["val"]
positive_index = np.flatnonzero(selected_model.classes_ == 1).item()
val_scores = selected_model.predict_proba(X_val)[:, positive_index]
val_predictions = (val_scores >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_val, val_predictions, labels=[0, 1]).ravel()
display(pd.DataFrame([[tn, fp], [fn, tp]], index=["Actual no", "Actual yes"],
                     columns=["Predicted no", "Predicted yes"]))
print(f"Missed subscribers: {fn:,} of {fn + tp:,} ({fn / (fn + tp):.1%}).")
print(f"False alarms: {fp:,} of {fp + tn:,} non-subscribers ({fp / (fp + tn):.1%}).")

,Predicted no,Predicted yes
Actual no,5403,78
Actual yes,537,159


Missed subscribers: 537 of 696 (77.2%).
False alarms: 78 of 5,481 non-subscribers (1.4%).


## Next experiments

The duration ablation can load the existing random split with `include_duration=True`. For a time-based comparison, first construct the chronological splits and fit a new preprocessor on the new training block. Pass those prepared `(X, y)` pairs to `run_experiment`, label the split and features, and compare ROC-AUC as well as average precision.